# ILI Component Detection — Inference

In [ ]:
import os
import sys
import copy
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.data import MetadataCatalog, DatasetCatalog, build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.structures import BoxMode
from detectron2.utils.visualizer import Visualizer
from detectron2.utils.logger import setup_logger
setup_logger()

# Add parent directory to path for importing project modules
sys.path.insert(0, os.path.abspath('..'))
from Dataset import register_detection_datasets
from Dataset.augmentations import apply_track_shift, apply_circular_roll
from utils import load_config

## Configuration & Model Setup

In [ ]:
CONFIG_PATH = "config.yaml"
CHECKPOINT_DIR = "../checkpoints"
MODEL_PATH = os.path.join(CHECKPOINT_DIR, "model_final.pth")

config = load_config(CONFIG_PATH)

# Register train/val/test datasets
n_total, n_train, n_val, n_test, num_classes, class_names = register_detection_datasets(config)
print(f"Dataset: {n_total} total | {n_train} train | {n_val} val | {n_test} test")
print(f"Classes ({num_classes}): {class_names}")

# Build Detectron2 config
cfg = get_cfg()
model_config = config.get('model_config', 'COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml')
cfg.merge_from_file(model_zoo.get_config_file(model_config))
cfg.OUTPUT_DIR = CHECKPOINT_DIR
cfg.DATASETS.TEST = ("data_detection_test",)
cfg.DATALOADER.NUM_WORKERS = config.get('num_workers', 4)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = num_classes
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = config.get('roi_heads_batch_size', 512)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = config.get('score_thresh_test', 0.6)
cfg.MODEL.WEIGHTS = MODEL_PATH
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

predictor = DefaultPredictor(cfg)
metadata = MetadataCatalog.get("data_detection_test")

print(f"\nModel loaded from: {MODEL_PATH}")
print(f"Score threshold: {cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST}")
print(f"Device: {cfg.MODEL.DEVICE}")

## Build Augmented Test Dataset

In `trainer.py`, the `ILIDatasetMapper` applies track shift and circular roll only
during training — the test/validation loaders use the default mapper (no augmentation).
This is standard practice: evaluate on clean data for unbiased metrics.

Here we additionally generate an augmented copy of the test set (both images **and**
annotations are transformed together) so we can measure model robustness under the
same augmentations used during training.

In [ ]:
AUG_TEST_DIR = "./augmented_test_images"
os.makedirs(AUG_TEST_DIR, exist_ok=True)

num_tracks = config.get('num_tracks', 22)
max_shift = config.get('max_track_shift', 25)
split_wrapped = config.get('split_wrapped_boxes', True)

# Load original test set once — reused throughout the notebook
test_dicts = DatasetCatalog.get("data_detection_test")
augmented_test_dicts = []

for d in test_dicts:
    im = cv2.imread(d["file_name"])
    h, w = im.shape[:2]
    annos = copy.deepcopy(d["annotations"])

    # Apply both augmentations to image AND annotations
    aug_im, aug_annos = apply_track_shift(im, annos, num_tracks, max_shift, h, w)
    aug_im, aug_annos = apply_circular_roll(aug_im, aug_annos, num_tracks, h, w, split_wrapped=split_wrapped)

    # Save augmented image to disk (PNG for lossless)
    basename = os.path.splitext(os.path.basename(d["file_name"]))[0]
    aug_path = os.path.join(AUG_TEST_DIR, f"{basename}.png")
    cv2.imwrite(aug_path, aug_im)

    augmented_test_dicts.append({
        "file_name": aug_path,
        "image_id": d["image_id"],
        "height": h,
        "width": w,
        "annotations": aug_annos,
    })

# Register augmented test dataset
AUG_DATASET = "data_detection_test_augmented"
if AUG_DATASET in DatasetCatalog:
    DatasetCatalog.remove(AUG_DATASET)
DatasetCatalog.register(AUG_DATASET, lambda: augmented_test_dicts)
MetadataCatalog.get(AUG_DATASET).set(thing_classes=class_names)

print(f"Augmented test dataset: {len(augmented_test_dicts)} images saved to {AUG_TEST_DIR}")

## Inference on Single Image

Load a local image, apply ILI augmentations (track shift + circular roll),
run inference on both original and augmented, and display side-by-side predictions.

In [ ]:
def apply_inference_augmentations(image, config):
    """
    Apply track shift and circular roll augmentations to an image for inference.

    The augmentation functions in Dataset/augmentations.py expect bounding box
    annotations alongside the image. For inference we only need the image
    transformation, so we pass empty annotations — the pixel-level shifts and
    rolls are independent of the bbox logic.

    Args:
        image: (H, W, C) uint8 numpy array.
        config: Config dict with augmentation parameters.

    Returns:
        Augmented image as (H, W, C) uint8 numpy array.
    """
    h, w = image.shape[:2]
    num_tracks = config.get('num_tracks', 22)
    max_shift = config.get('max_track_shift', 25)
    split_wrapped = config.get('split_wrapped_boxes', True)

    augmented, _ = apply_track_shift(image, [], num_tracks, max_shift, h, w)
    augmented, _ = apply_circular_roll(augmented, [], num_tracks, h, w, split_wrapped=split_wrapped)

    return augmented

In [ ]:
# --- Set image path (use a test sample by default) ---
IMAGE_PATH = random.choice(test_dicts)["file_name"]  # or replace with your own local path
print(f"Image: {IMAGE_PATH}")

im = cv2.imread(IMAGE_PATH)
assert im is not None, f"Cannot read image: {IMAGE_PATH}"

# Apply augmentations
im_aug = apply_inference_augmentations(im, config)

# Run inference on both
outputs_orig = predictor(im)
outputs_aug = predictor(im_aug)

# Visualize
v_orig = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.5)
out_orig = v_orig.draw_instance_predictions(outputs_orig["instances"].to("cpu"))

v_aug = Visualizer(im_aug[:, :, ::-1], metadata=metadata, scale=0.5)
out_aug = v_aug.draw_instance_predictions(outputs_aug["instances"].to("cpu"))

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
axes[0].imshow(out_orig.get_image())
axes[0].set_title("Original — Predictions")
axes[0].axis("off")
axes[1].imshow(out_aug.get_image())
axes[1].set_title("Augmented (track shift + circular roll) — Predictions")
axes[1].axis("off")
plt.tight_layout()
plt.show()

instances_orig = outputs_orig["instances"].to("cpu")
instances_aug = outputs_aug["instances"].to("cpu")
print(f"Original:  {len(instances_orig)} detections")
print(f"Augmented: {len(instances_aug)} detections")

## Evaluation on Test Set

COCO evaluation on both the original and augmented test sets.

In [ ]:
TEST_OUTPUT_DIR = "./test_output"
AUG_TEST_OUTPUT_DIR = "./test_output_augmented"
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
os.makedirs(AUG_TEST_OUTPUT_DIR, exist_ok=True)

# --- Original test set ---
print("=" * 60)
print("Original Test Set")
print("=" * 60)
evaluator = COCOEvaluator("data_detection_test", output_dir=TEST_OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, "data_detection_test")
results_orig = inference_on_dataset(predictor.model, test_loader, evaluator)
print(results_orig)

# --- Augmented test set ---
print("\n" + "=" * 60)
print("Augmented Test Set (track shift + circular roll)")
print("=" * 60)
evaluator_aug = COCOEvaluator(AUG_DATASET, output_dir=AUG_TEST_OUTPUT_DIR)
test_loader_aug = build_detection_test_loader(cfg, AUG_DATASET)
results_aug = inference_on_dataset(predictor.model, test_loader_aug, evaluator_aug)
print(results_aug)

## Test Set Predictions

Visualize original predictions, augmented predictions, and ground truth side-by-side.

In [ ]:
indices = random.sample(range(len(test_dicts)), min(10, len(test_dicts)))

for idx in indices:
    d_orig = test_dicts[idx]
    d_aug = augmented_test_dicts[idx]

    im_orig = cv2.imread(d_orig["file_name"])
    im_aug = cv2.imread(d_aug["file_name"])

    out_orig = predictor(im_orig)
    out_aug = predictor(im_aug)

    v_orig = Visualizer(im_orig[:, :, ::-1], metadata=metadata, scale=0.5)
    v_aug = Visualizer(im_aug[:, :, ::-1], metadata=metadata, scale=0.5)
    v_gt = Visualizer(im_orig[:, :, ::-1], metadata=metadata, scale=0.5)

    vis_orig = v_orig.draw_instance_predictions(out_orig["instances"].to("cpu"))
    vis_aug = v_aug.draw_instance_predictions(out_aug["instances"].to("cpu"))
    vis_gt = v_gt.draw_dataset_dict(d_orig)

    fig, axes = plt.subplots(1, 3, figsize=(30, 6))
    axes[0].imshow(vis_orig.get_image())
    axes[0].set_title("Original — Predictions")
    axes[0].axis("off")
    axes[1].imshow(vis_aug.get_image())
    axes[1].set_title("Augmented — Predictions")
    axes[1].axis("off")
    axes[2].imshow(vis_gt.get_image())
    axes[2].set_title("Ground Truth")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## False Negative Analysis

Compute false negatives (unmatched GT at IoU < 0.5) on both original and
augmented test sets, then visualize a sample of the FN images.

In [ ]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / float(areaA + areaB - inter) if (areaA + areaB - inter) > 0 else 0.0


def find_false_negatives(dataset_dicts, predictor, class_names, iou_thresh=0.5):
    """Return (fn_count, total_gt, per_category_fn, fn_image_indices)."""
    fn_count = 0
    total_gt = 0
    cat_fn = {name: 0 for name in class_names}
    fn_indices = set()

    for i, d in enumerate(dataset_dicts):
        im = cv2.imread(d["file_name"])
        outputs = predictor(im)
        pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()

        for ann in d["annotations"]:
            total_gt += 1
            gt_box = BoxMode.convert(ann["bbox"], BoxMode.XYWH_ABS, BoxMode.XYXY_ABS)
            ious = [compute_iou(gt_box, pb) for pb in pred_boxes]
            if not ious or max(ious) < iou_thresh:
                fn_count += 1
                cat_fn[class_names[ann["category_id"]]] += 1
                fn_indices.add(i)

    return fn_count, total_gt, cat_fn, sorted(fn_indices)


# --- Original test set ---
print("--- Original Test Set ---")
fn_orig, gt_orig, cat_fn_orig, fn_idx_orig = find_false_negatives(
    test_dicts, predictor, class_names,
)
print(f"False negatives: {fn_orig} / {gt_orig} (IoU < 0.5)")
print(f"Per-category: {cat_fn_orig}")

# --- Augmented test set ---
print("\n--- Augmented Test Set ---")
fn_aug, gt_aug, cat_fn_aug, fn_idx_aug = find_false_negatives(
    augmented_test_dicts, predictor, class_names,
)
print(f"False negatives: {fn_aug} / {gt_aug} (IoU < 0.5)")
print(f"Per-category: {cat_fn_aug}")

In [ ]:
# Visualize FN images: original predictions | augmented predictions | ground truth
MAX_VIS = 10
fn_all = sorted(set(fn_idx_orig) | set(fn_idx_aug))[:MAX_VIS]

for idx in fn_all:
    d_orig = test_dicts[idx]
    d_aug = augmented_test_dicts[idx]

    im_orig = cv2.imread(d_orig["file_name"])
    im_aug = cv2.imread(d_aug["file_name"])

    out_orig = predictor(im_orig)
    out_aug = predictor(im_aug)

    v_orig = Visualizer(im_orig[:, :, ::-1], metadata=metadata, scale=0.5)
    v_aug = Visualizer(im_aug[:, :, ::-1], metadata=metadata, scale=0.5)
    v_gt = Visualizer(im_orig[:, :, ::-1], metadata=metadata, scale=0.5)

    vis_orig = v_orig.draw_instance_predictions(out_orig["instances"].to("cpu"))
    vis_aug = v_aug.draw_instance_predictions(out_aug["instances"].to("cpu"))
    vis_gt = v_gt.draw_dataset_dict(d_orig)

    fig, axes = plt.subplots(1, 3, figsize=(30, 6))
    axes[0].imshow(vis_orig.get_image())
    axes[0].set_title("Original — Predictions")
    axes[0].axis("off")
    axes[1].imshow(vis_aug.get_image())
    axes[1].set_title("Augmented — Predictions")
    axes[1].axis("off")
    axes[2].imshow(vis_gt.get_image())
    axes[2].set_title("Ground Truth")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## Best Threshold Search

Sweep score thresholds on the original test set to find the best AP50,
then report augmented performance at that threshold.

In [ ]:
best_ap50 = -1
best_threshold = -1

for threshold in np.arange(0.1, 1.0, 0.05):
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(threshold)
    predictor = DefaultPredictor(cfg)

    evaluator = COCOEvaluator("data_detection_test", output_dir=TEST_OUTPUT_DIR)
    test_loader = build_detection_test_loader(cfg, "data_detection_test")
    metrics = inference_on_dataset(predictor.model, test_loader, evaluator)

    ap50 = metrics["bbox"]["AP50"]
    print(f"  threshold={threshold:.2f}  AP50={ap50:.3f}")

    if ap50 > best_ap50:
        best_ap50 = ap50
        best_threshold = threshold

print(f"\nBest AP50: {best_ap50:.3f} at threshold: {best_threshold:.2f}")

# Report augmented performance at the best threshold
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(best_threshold)
predictor = DefaultPredictor(cfg)

evaluator_aug = COCOEvaluator(AUG_DATASET, output_dir=AUG_TEST_OUTPUT_DIR)
test_loader_aug = build_detection_test_loader(cfg, AUG_DATASET)
results_aug_best = inference_on_dataset(predictor.model, test_loader_aug, evaluator_aug)
print(f"\nAugmented test set at best threshold ({best_threshold:.2f}):")
print(results_aug_best)